# 01 · Auditoria de les dades i join vols + meteorologia

**Fase 1 del projecte: comprensió de les dades** [T: T1, L1 "First audit of a dataset"]

Objectiu: saber què representa cada fila i cada columna **abans** de netejar res.
Aquest notebook:

1. llegeix les taules del dataset de Kaggle *US 2023 Civil Flights, delays, meteo and aircrafts*;
2. en fa una auditoria (tipus, nuls, rangs, duplicats, cobertura temporal);
3. comprova si `deptime_label` i `flight_duration` es basen en valors **programats** o **reals**: si són reals, hi hauria fuita de dades;
4. fa el join amb la meteorologia de l'aeroport d'origen i del de destinació, i en comprova la granularitat;
5. desa la taula unida, **sense netejar**, per al notebook següent.

No hi ha cap pas aleatori, així que el resultat és sempre el mateix.

## 0. Configuració

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Rutes: posa els CSV de Kaggle a DATA_DIR
DATA_DIR = Path("../data/raw")
OUT_DIR = Path("../data/interim")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FLIGHTS_CSV = DATA_DIR / "US_flights_2023.csv"
WEATHER_CSV = DATA_DIR / "weather_meteo_by_airport.csv"
CANCELLED_CSV = DATA_DIR / "Cancelled_Diverted_2023.csv"   # opcional
AIRPORTS_CSV = DATA_DIR / "airports_geolocation.csv"       # opcional

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.2f}".format)
print("pandas", pd.__version__)

## 1. Lectura de les dades

La taula de vols té uns 6,7 milions de files. Per estalviar memòria, les columnes de text
es llegeixen com a `category`. També es passen tots els noms de columna a minúscules,
perquè el codi no depengui de com estiguin escrits a l'original.

In [ ]:
def load_csv(path):
    # Llegeix un CSV amb les columnes de text com a 'category' i noms en minúscules.
    sample = pd.read_csv(path, nrows=20_000)              # per detectar quines columnes són text
    text_cols = [c for c in sample.columns if not pd.api.types.is_numeric_dtype(sample[c])]
    df = pd.read_csv(path, dtype={c: "category" for c in text_cols})
    df.columns = [c.strip().lower() for c in df.columns]
    return df

fl = load_csv(FLIGHTS_CSV)
we = load_csv(WEATHER_CSV)

# Dates com a datetime (es fan servir per al join i per a la partició temporal)
fl["flightdate"] = pd.to_datetime(fl["flightdate"].astype(str))
we["time"] = pd.to_datetime(we["time"].astype(str))

print("Vols:", fl.shape, " · Meteo:", we.shape)
fl.head()

In [ ]:
we.head()

## 2. Auditoria genèrica de cada taula  [T: L1]

Per a cada columna mirem el tipus, els nuls, quants valors diferents té i un exemple.
Això serveix per decidir el **tipus semàntic** de cada variable (numèrica, categòrica,
data o identificador) i és la base de la secció 4.1 de la memòria.

In [ ]:
def audit_table(df):
    # Taula resum per columna + nombre de files duplicades exactes.
    info = pd.DataFrame({
        "tipus": df.dtypes.astype(str),
        "nuls": df.isna().sum(),
        "% nuls": 100 * df.isna().mean(),
        "valors diferents": df.nunique(dropna=True),
        "exemple": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })
    mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Files: {len(df):,} · Columnes: {df.shape[1]} · Memòria: {mem:,.0f} MB · "
          f"Duplicades exactes: {df.duplicated().sum():,}")
    return info

audit_table(fl)

In [ ]:
# Resum de les numèriques: percentils 1 % i 99 % per veure valors extrems
fl.describe(percentiles=[.01, .25, .5, .75, .99]).T

## 3. Comprovacions específiques dels vols

### 3.1 Cobertura temporal
Ens cal per decidir la partició train/test (temporal o per dies) i per veure si falta algun mes.

In [ ]:
print("Rang:", fl.flightdate.min().date(), "→", fl.flightdate.max().date(),
      "·", fl.flightdate.dt.normalize().nunique(), "dies diferents")
fl.groupby(fl.flightdate.dt.month).size().rename("vols").rename_axis("mes").to_frame().T

### 3.2 Candidats a target i desbalanceig
Proporció de vols endarrerits segons diferents llindars del retard d'arribada. La definició
oficial del Departament de Transport dels EUA és **≥ 15 minuts**.

In [ ]:
a = fl["arr_delay"]
pd.DataFrame({
    "% vols endarrerits": [100 * (a > 0).mean(), 100 * (a >= 15).mean(), 100 * (a >= 60).mean()]
}, index=["> 0 min", ">= 15 min", ">= 60 min"])

Com defineix el dataset les categories de `arr_delay_type` i `dep_delay_type`:

In [ ]:
for col, base in [("arr_delay_type", "arr_delay"), ("dep_delay_type", "dep_delay")]:
    display(fl.groupby(col, observed=True)[base].agg(["count", "min", "max"]))

### 3.3 `deptime_label`: hora programada o hora real?
Si l'etiqueta es basa en l'hora **real** de sortida, un vol molt endarrerit passa a una
franja més tardana. En aquest cas, la franja contindria informació del retard, és a dir,
**fuita de dades**.

Prova: si la distribució de franges es desplaça cap a *Evening*/*Night* quan creix el retard,
és sospitós. Una altra prova és a la secció 5, amb els vols cancel·lats.

In [ ]:
bucket = pd.cut(fl["dep_delay"], bins=[-np.inf, 0, 15, 60, 180, np.inf],
                labels=["<=0", "1-15", "16-60", "61-180", ">180"])
(pd.crosstab(bucket, fl["deptime_label"], normalize="index") * 100).rename_axis("dep_delay (min)")

### 3.4 `flight_duration`: durada programada o real?
Si és la durada **real**, es compleix que real − programada = `arr_delay` − `dep_delay`.
Per tant, `flight_duration − (arr_delay − dep_delay)` seria la durada programada i hauria de
ser **gairebé constant dins de cada ruta** (companyia, origen, destinació).

- Si la segona desviació és molt més petita que la primera, `flight_duration` és la durada
  real, que és fuita. Però llavors podem **reconstruir la durada programada**, que sí que es
  coneix abans del vol.
- Si són semblants, `flight_duration` ja és la programada.

In [ ]:
g = pd.DataFrame({
    "flight_duration": fl["flight_duration"],
    "implicita_programada": fl["flight_duration"] - (fl["arr_delay"] - fl["dep_delay"]),
    "airline": fl["airline"], "o": fl["dep_airport"], "d": fl["arr_airport"],
})
stds = g.groupby(["airline", "o", "d"], observed=True)[["flight_duration", "implicita_programada"]].std()
stds.median().rename("desviació típica mediana per ruta (min)").to_frame()

### 3.5 Valors impossibles o sospitosos i duplicats  [T: L1 "Finding errors"]
Aquí només es **compten**. Què se'n fa es decidirà a la fase de neteja: corregir, marcar com
a nul o eliminar.

In [ ]:
pd.Series({
    "aicraft_age < 0 o > 60": ((fl.aicraft_age < 0) | (fl.aicraft_age > 60)).sum(),
    "flight_duration <= 0": (fl.flight_duration <= 0).sum(),
    "origen == destinació": (fl.dep_airport.astype(str) == fl.arr_airport.astype(str)).sum(),
    "duplicats (data, avió, origen, destinació, franja)":
        fl.duplicated(subset=["flightdate", "tail_number", "dep_airport",
                              "arr_airport", "deptime_label"]).sum(),
}, name="files").to_frame()

## 4. Auditoria de la meteorologia

Ha d'haver-hi **una sola fila per (aeroport, dia)**. Si no és així, el join multiplicaria
els vols i canviaria el significat de cada fila [T: L2 "Joining tables: be careful with granularity"].

In [ ]:
audit_table(we)

In [ ]:
n_dup = we.duplicated(subset=["airport_id", "time"]).sum()
possibles = we.airport_id.nunique() * we.time.nunique()
print(f"Files repetides per (aeroport, dia): {n_dup:,}  (han de ser 0)")
print(f"Aeroports: {we.airport_id.nunique():,} · Dies: {we.time.nunique():,} · "
      f"Rang: {we.time.min().date()} → {we.time.max().date()}")
print(f"Combinacions aeroport×dia presents: {len(we):,} de {possibles:,} ({100*len(we)/possibles:.1f} %)")

In [ ]:
# Valors físicament impossibles
pd.Series({
    "tmin > tmax": (we.tmin > we.tmax).sum(),
    "tavg fora de [tmin, tmax]": ((we.tavg < we.tmin) | (we.tavg > we.tmax)).sum(),
    "prcp < 0": (we.prcp < 0).sum(),
    "snow < 0": (we.snow < 0).sum(),
    "wspd < 0": (we.wspd < 0).sum(),
    "wdir fora de [0, 360]": ((we.wdir < 0) | (we.wdir > 360)).sum(),
    "pres fora de [870, 1085] hPa": ((we.pres < 870) | (we.pres > 1085)).sum(),
}, name="files").to_frame()

## 5. Taules opcionals (només per auditar)

- **Vols cancel·lats o desviats.** No entren al model perquè no tenen retard d'arribada, però
  serveixen per comprovar `deptime_label`. Un vol cancel·lat no té hora real de sortida: si
  gairebé tots tenen franja informada, l'etiqueta ve de l'hora **programada**.
- **Geolocalització d'aeroports.** Serviria per calcular la distància exacta i l'estat de
  cada aeroport, si decidim fer-la servir.

In [ ]:
if CANCELLED_CSV.exists():
    cd = load_csv(CANCELLED_CSV)
    display(audit_table(cd))
    canc = cd[cd["cancelled"] == 1]
    print(f"Cancel·lats: {len(canc):,} · amb deptime_label informat: "
          f"{100 * canc['deptime_label'].notna().mean():.1f} %")
else:
    print("No hi ha", CANCELLED_CSV.name)

if AIRPORTS_CSV.exists():
    display(audit_table(load_csv(AIRPORTS_CSV)))
else:
    print("No hi ha", AIRPORTS_CSV.name)

## 6. Join vols + meteorologia (origen i destinació)

- La clau és (aeroport, dia). La meteorologia s'uneix **dues vegades**: una amb `dep_airport`
  (columnes `dep_*`) i una altra amb `arr_airport` (columnes `arr_*`).
- És un *left join*: cap vol es perd. Els que no tenen meteorologia queden amb nuls, que es
  tractaran a la fase de neteja.
- `validate="many_to_one"` fa que pandas doni error si la meteorologia té (aeroport, dia)
  repetits.

In [ ]:
w_cols = [c for c in we.columns if c not in ("airport_id", "time")]
w = we.assign(airport_id=we["airport_id"].astype(str), time=we["time"].dt.normalize())

# Cobertura de claus abans d'unir: quins aeroports no tenen meteorologia?
w_ids = set(w["airport_id"])
for side in ["dep_airport", "arr_airport"]:
    falten = sorted(set(fl[side].astype(str)) - w_ids)
    print(f"{side}: {fl[side].nunique()} aeroports · sense meteo: {len(falten)} {falten[:15]}")

In [ ]:
df = fl.copy()
df["_date"] = df["flightdate"].dt.normalize()
n0 = len(df)

for side, prefix in [("dep_airport", "dep_"), ("arr_airport", "arr_")]:
    right = w.rename(columns={c: prefix + c for c in w_cols})
    df = (df.assign(_key=df[side].astype(str))
            .merge(right, how="left", left_on=["_key", "_date"],
                   right_on=["airport_id", "time"], validate="many_to_one")
            .drop(columns=["airport_id", "time", "_key"]))
    print(f"Vols sense meteo a {side}: {100 * df[prefix + w_cols[0]].isna().mean():.2f} %")

df = df.drop(columns="_date")
assert len(df) == n0, "El join ha canviat el nombre de files!"
print(f"Files abans: {n0:,} · després: {len(df):,} → una fila continua sent un vol ✔")

In [ ]:
# Nuls de les columnes meteorològiques després del join
met = [p + c for p in ("dep_", "arr_") for c in w_cols]
(100 * df[met].isna().mean()).rename("% nuls").to_frame().T

## 7. Desar la taula unida (encara sense netejar)

In [ ]:
try:
    dest = OUT_DIR / "flights_weather_2023.parquet"
    df.to_parquet(dest, index=False)        # necessita pyarrow
except ImportError:
    dest = OUT_DIR / "flights_weather_2023.pkl"
    df.to_pickle(dest)                       # alternativa si no hi ha pyarrow
print(f"Desat a {dest} · {df.shape[0]:,} files × {df.shape[1]} columnes")

## 8. Conclusions de l'auditoria (a omplir pel grup)

| Pregunta | Resultat | Decisió |
|---|---|---|
| `deptime_label` és l'hora programada o la real? (3.3 i 5) | | |
| `flight_duration` és la durada programada o la real? (3.4) | | |
| Proporció de la classe positiva amb el llindar escollit (3.2) | | |
| Aeroports sense meteorologia i % de vols afectats (6) | | |
| Columnes meteorològiques amb molts nuls (4 i 6) | | |
| Valors impossibles i duplicats (3.5 i 4) | | |
| Mesos coberts i equilibri entre mesos (3.1) | | |